# Model adequacy and residual diagnostics: lab notebook

**Statistical Modelling and Forecasting, Regression and exponential smoothing**

Run the cells from top to bottom (Runtime > Run all). NumPy, pandas, SciPy, statsmodels and scikit-learn are already installed in Colab. Read each section, run it, and compare your output with the expected output at the end of the notebook.

In [ ]:
import warnings
warnings.filterwarnings("ignore")  # keeps the output tidy

In [ ]:
import numpy as np
import pandas as pd
import statsmodels.api as sm
from statsmodels.stats.stattools import durbin_watson, jarque_bera
from statsmodels.stats.diagnostic import het_breuschpagan, acorr_breusch_godfrey
from statsmodels.stats.outliers_influence import variance_inflation_factor

### 1. A regression of log consumption on log income (levels) looks excellent...

In [ ]:
# 1. A regression of log consumption on log income (levels) looks excellent...
d = sm.datasets.macrodata.load_pandas().data
y = np.log(d["realcons"]); x = np.log(d["realdpi"])
X = sm.add_constant(x)
fit = sm.OLS(y, X).fit()
print("R2 =", round(fit.rsquared, 4), "| slope =", round(fit.params.iloc[1], 3))

### 2. ... but the residual diagnostics say otherwise

In [ ]:
# 2. ... but the residual diagnostics say otherwise
res = fit.resid
print("Durbin-Watson:", round(durbin_watson(res), 3), "-> strong positive autocorrelation (r1 about", round(1 - durbin_watson(res) / 2, 2), ")")
bg = acorr_breusch_godfrey(fit, nlags=4)
print("Breusch-Godfrey LM p-value:", round(bg[1], 5))
bp = het_breuschpagan(res, X)
print("Breusch-Pagan LM =", round(bp[0], 2), "p =", round(bp[1], 4))
jb = jarque_bera(res)
print("Jarque-Bera =", round(jb[0], 2), "p =", round(jb[1], 4), "| skew =", round(jb[2], 2), "kurtosis =", round(jb[3], 2))

### 3. Leverage, standardised residuals and Cook's distance

In [ ]:
# 3. Leverage, standardised residuals and Cook's distance
inf = fit.get_influence()
h = inf.hat_matrix_diag
r = inf.resid_studentized_internal
cook = inf.cooks_distance[0]
print("average leverage p/n =", round(2 / len(y), 4), "| largest leverage:", round(h.max(), 4), "at quarter", int(h.argmax()))
print("largest |standardised residual|:", round(np.abs(r).max(), 2), "| largest Cook's D:", round(cook.max(), 4), "| 4/n =", round(4 / len(y), 4))

### 4. Differencing removes most of the autocorrelation

In [ ]:
# 4. Differencing removes most of the autocorrelation
dg = 100 * np.log(d[["realcons", "realdpi"]]).diff().dropna()
f2 = sm.OLS(dg["realcons"], sm.add_constant(dg["realdpi"])).fit()
print("growth-rate regression: R2 =", round(f2.rsquared, 3), "| Durbin-Watson =", round(durbin_watson(f2.resid), 2))

### 5. Multicollinearity: Longley data (VIF)

In [ ]:
# 5. Multicollinearity: Longley data (VIF)
L = sm.datasets.longley.load_pandas().data
XL = sm.add_constant(L[["GNPDEFL", "GNP", "UNEMP", "ARMED", "POP", "YEAR"]])
vif = pd.Series([variance_inflation_factor(XL.values, i) for i in range(1, XL.shape[1])], index=XL.columns[1:])
print("VIF:"); print(vif.round(1).to_string())
print("VIF = 1/(1-R2): R2 = 0.95 gives", 1 / (1 - 0.95))

### 6. The numerical problem: leverage and Cook's distance

In [ ]:
# 6. The numerical problem: leverage and Cook's distance
n, Sxx, xb, xi, ei, s = 12, 100, 15, 21, 4, 2.0
hh = 1 / n + (xi - xb) ** 2 / Sxx
rs = ei / (s * np.sqrt(1 - hh))
print("leverage:", round(hh, 4), "| standardised residual:", round(rs, 3), "| Cook's D:", round(rs ** 2 * hh / (2 * (1 - hh)), 3))
e = np.array([2, 3, 1, -1, -2, -3, -1, 1])
print("Durbin-Watson of the eight residuals:", round(durbin_watson(e), 3))

## Expected output

Running all cells above should print:

```
R2 = 0.9984 | slope = 1.032
Durbin-Watson: 0.188 -> strong positive autocorrelation (r1 about 0.91 )
Breusch-Godfrey LM p-value: 0.0
Breusch-Pagan LM = 5.96 p = 0.0146
Jarque-Bera = 3.29 p = 0.1934 | skew = -0.28 kurtosis = 2.72
average leverage p/n = 0.0099 | largest leverage: 0.0229 at quarter 0
largest |standardised residual|: 2.57 | largest Cook's D: 0.0548 | 4/n = 0.0197
growth-rate regression: R2 = 0.193 | Durbin-Watson = 1.89
VIF:
GNPDEFL     135.5
GNP        1788.5
UNEMP        33.6
ARMED         3.6
POP         399.2
YEAR        759.0
VIF = 1/(1-R2): R2 = 0.95 gives 19.999999999999982
leverage: 0.4433 | standardised residual: 2.681 | Cook's D: 2.861
Durbin-Watson of the eight residuals: 0.633
```

## Practice exercises

Try these in the cells below.

1. Choose "Curved relationship". Which plot shows the problem most clearly?
2. Choose "Autocorrelated errors" and slide ρ from 0 to 0.9. Read DW and describe the residuals over time.
3. Choose "One influential point" and compare the slope with and without it (toggle the point).

In [ ]:
# Exercise 1


In [ ]:
# Exercise 2


In [ ]:
# Exercise 3
